# [Starter EDA + Offline RDKit 2026.3.3 Wheels + MS/MS Benchmark] Adduct MRR@25 & Dual-Channel Cosine Diagnostics

Hi everyone,

To help participants working on offline submission pipelines and adduct-specific retrieval analysis in **Enveda CASMI 2026**, I have published a clean **Public EDA & Starter Notebook** along with an **Offline RDKit 2026.3.3 Wheelhouse Dataset** and a **Tandem MS/MS Retrieval Evaluation Benchmark**:

1. **Public Competition EDA Notebook:**  
   👉 [CASMI 2026 MS/MS Spectra EDA & RDKit Starter](https://www.kaggle.com/code/thivnti/casmi-2026-msms-spectra-eda-rdkit-starter)
2. **Offline RDKit 2026.3.3 Linux Wheels (`cp312` / `cp313`):**  
   👉 [CASMI 2026 Offline RDKit Wheels Dataset](https://www.kaggle.com/datasets/thivnti/casmi2026-offline-rdkit)
3. **MS/MS Spectral Retrieval Evaluation Benchmark (`9.41/10` Usability):**  
   👉 [Mass Spectrometry Retrieval Evaluation Benchmark](https://www.kaggle.com/datasets/thivnti/mass-spectrometry-retrieval-evaluation-benchmark)
4. **PyTorch Dual-Band 4096-Bit Fingerprint Projector Model:**  
   👉 [MS/MS Dual-Band Spectral Fingerprint Projector](https://www.kaggle.com/models/thivnti/msms-spectral-fingerprint-projector)

### Key Takeaways from the EDA
- **Offline RDKit Installation Snippet:**
  `!pip install --no-index --find-links=/kaggle/input/casmi2026-offline-rdkit rdkit`
- **Adduct Sensitivity:** Combining direct `sqrt-L2` fragment cosine with neutral-loss alignment significantly improves retrieval rank on `[M+Na]+` and `[M+NH4]+` precursor species compared to single-channel cosine.

Hope this helps speed up your experiments! Feedback and upvotes on the notebook/datasets are very welcome if you find them useful.

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

def find_input_files(substring=None):
    matches = []
    for root, _, files in os.walk('/kaggle/input', followlinks=True):
        for fname in files:
            if substring is None or substring in fname:
                matches.append(Path(root) / fname)
    return sorted(matches)

all_files = find_input_files()
print(f"Total mounted input files discovered: {len(all_files)}")


In [ ]:
ms_csv = find_input_files('msms_retrieval_benchmark.csv')[0]
df_ms = pd.read_csv(ms_csv)
adduct_stats = df_ms.groupby(['ionization_mode', 'adduct_type']).agg(
    spectra=('spectrum_group_id', 'count'),
    mean_mrr=('reciprocal_rank', 'mean'),
    mean_cosine=('cosine_similarity_top1', 'mean'),
    mean_neutral_loss=('neutral_loss_score', 'mean')
).reset_index().round(4)
display(adduct_stats)
adduct_stats.to_csv('casmi26_writeup_adduct_diagnostics.csv', index=False)